In [6]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv("data/insurance.csv")

print("Dataset Preview:")
display(df.head())
print("\nDataset Info:")
df.info()

Dataset Preview:


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [ ]:
num_cols = ['age', 'bmi', 'children', 'charges']
desc_stats = pd.DataFrame(index=num_cols)

desc_stats['Mean'] = df[num_cols].mean()
desc_stats['Median'] = df[num_cols].median()
desc_stats['Std Dev'] = df[num_cols].std()
desc_stats['IQR'] = df[num_cols].apply(lambda x: stats.iqr(x))
desc_stats['Skewness'] = df[num_cols].apply(lambda x: stats.skew(x))
desc_stats['Kurtosis'] = df[num_cols].apply(lambda x: stats.kurtosis(x))

print("--- Descriptive Metrics ---")
display(desc_stats.round(3))

print("\n--- Hypothesis Test 1: Smokers vs. Non-Smokers (Charges) ---")
smokers = df[df['smoker'] == 'yes']['charges']
non_smokers = df[df['smoker'] == 'no']['charges']

# Check Normality (Shapiro-Wilk)
shapiro_smokers = stats.shapiro(smokers)
shapiro_non_smokers = stats.shapiro(non_smokers)
print(f"Shapiro-Wilk Smokers p-value: {shapiro_smokers.pvalue:.4e}")
print(f"Shapiro-Wilk Non-Smokers p-value: {shapiro_non_smokers.pvalue:.4e}")

# Check Equal Variance (Levene's Test)
levene_test = stats.levene(smokers, non_smokers)
print(f"Levene's Test p-value: {levene_test.pvalue:.4e}")

# Run Appropriate Test
alpha = 0.05
if shapiro_smokers.pvalue > alpha and shapiro_non_smokers.pvalue > alpha and levene_test.pvalue > alpha:
    stat, p_val = stats.ttest_ind(smokers, non_smokers, equal_var=True)
    test_name = "Two-Sample t-test"
else:
    stat, p_val = stats.mannwhitneyu(smokers, non_smokers)
    test_name = "Mann-Whitney U Test"

conclusion = "Reject H0 (Significant difference)" if p_val < alpha else "Fail to Reject H0 (No significant difference)"
print(f"\nApplied Test: {test_name}")
print(f"Statistic: {stat:.4f}, p-value: {p_val:.4e}")
print(f"Conclusion at alpha=0.05: {conclusion}")

print("\n--- Hypothesis Test 2: One-Way ANOVA across Regions (Charges) ---")
regions = df['region'].unique()
region_groups = [df[df['region'] == r]['charges'] for r in regions]

f_stat, f_pval = stats.f_oneway(*region_groups)
anova_conclusion = "Reject H0" if f_pval < alpha else "Fail to Reject H0"

print(f"F-Statistic: {f_stat:.4f}, p-value: {f_pval:.4f}")
print(f"Conclusion at alpha=0.05: {anova_conclusion}")

--- Descriptive Metrics ---


,Mean,Median,Std Dev,IQR,Skewness,Kurtosis
age,39.207,39.000,14.050,24.000,0.056,-1.245
bmi,30.663,30.400,6.098,8.398,0.284,-0.055
children,1.095,1.000,1.205,2.000,0.937,0.197
charges,13270.422,9382.033,12110.011,11899.625,1.514,1.596



--- Hypothesis Test 1: Smokers vs. Non-Smokers (Charges) ---
Shapiro-Wilk Smokers p-value: 3.6250e-09
Shapiro-Wilk Non-Smokers p-value: 1.4459e-28
Levene's Test p-value: 1.5593e-66

Applied Test: Mann-Whitney U Test
Statistic: 284133.0000, p-value: 5.2702e-130
Conclusion at alpha=0.05: Reject H0 (Significant difference)

--- Hypothesis Test 2: One-Way ANOVA across Regions (Charges) ---
F-Statistic: 2.9696, p-value: 0.0309
Conclusion at alpha=0.05: Reject H0


In [ ]:
formula = "charges ~ age + bmi + children + C(sex) + C(smoker) + C(region)"
ols_model = smf.ols(formula=formula, data=df).fit()

print(ols_model.summary())

# Multicollinearity Check: Variance Inflation Factor (VIF)
continuous_vars = df[['age', 'bmi', 'children']].copy()
continuous_vars['Intercept'] = 1  # Required for VIF calculation

vif_data = pd.DataFrame()
vif_data["Variable"] = continuous_vars.columns
vif_data["VIF"] = [variance_inflation_factor(continuous_vars.values, i) for i in range(continuous_vars.shape[1])]

print("\n--- Variance Inflation Factor (VIF) ---")
display(vif_data[vif_data["Variable"] != 'Intercept'])

# Gauss-Markov Residual Diagnostics Plots
import matplotlib.pyplot as plt

fitted_vals = ols_model.fittedvalues
residuals = ols_model.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Residuals vs Fitted
axes[0].scatter(fitted_vals, residuals, alpha=0.4, edgecolors='k')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Fitted (Linearity & Homoscedasticity)')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')

# Plot 2: Normal Q-Q Plot
sm.qqplot(residuals, line='45', fit=True, ax=axes[1])
axes[1].set_title('Normal Q-Q Plot (Normality of Residuals)')

plt.tight_layout()
plt.show()

# Jarque-Bera Normality Test
jb_stat, jb_pval, _, _ = sm.stats.stattools.jarque_bera(residuals)
print(f"Jarque-Bera Test Statistic: {jb_stat:.2f}, p-value: {jb_pval:.4e}")

                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.751
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     500.8
Date:                Sun, 13 Sep 2026   Prob (F-statistic):               0.00
Time:                        17:43:28   Log-Likelihood:                -13548.
No. Observations:                1338   AIC:                         2.711e+04
Df Residuals:                    1329   BIC:                         2.716e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept              -1.19

,Variable,VIF
0,age,1.013816
1,bmi,1.012152
2,children,1.001874


Jarque-Bera Test Statistic: 718.89, p-value: 7.8635e-157


C:\Users\Admin\AppData\Local\Temp\ipykernel_20888\2320431658.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.express as px
import matplotlib.pyplot as plt

st.set_page_config(page_title="Insurance Statistical Dashboard", layout="wide")

# Cache data loading
@st.cache_data
def load_data():
    return pd.read_csv("data/insurance.csv")

df = load_data()

st.title("Applied Statistical Modeling & Interactive Web Dashboard")

# Navigation Tabs
tab1, tab2, tab3 = st.tabs([
    "Data Exploration", 
    "Hypothesis Testing Lab", 
    "Live Prediction & Diagnostics"
])

with tab1:
    st.header("Exploratory Data Analysis")
    
    # Sidebar Filters
    st.sidebar.subheader("Filter Data (Tab 1)")
    age_range = st.sidebar.slider("Age Range", int(df['age'].min()), int(df['age'].max()), (20, 60))
    selected_regions = st.sidebar.multiselect("Select Regions", options=df['region'].unique(), default=list(df['region'].unique()))
    selected_smoker = st.sidebar.multiselect("Smoking Status", options=df['smoker'].unique(), default=list(df['smoker'].unique()))
    
    filtered_df = df[
        (df['age'] >= age_range[0]) & 
        (df['age'] <= age_range[1]) & 
        (df['region'].isin(selected_regions)) & 
        (df['smoker'].isin(selected_smoker))
    ]
    
    col1, col2 = st.columns([1, 2])
    with col1:
        st.subheader("Summary Metrics")
        num_cols = ['age', 'bmi', 'children', 'charges']
        desc = pd.DataFrame(index=num_cols)
        desc['Mean'] = filtered_df[num_cols].mean()
        desc['Median'] = filtered_df[num_cols].median()
        desc['Std'] = filtered_df[num_cols].std()
        desc['IQR'] = filtered_df[num_cols].apply(lambda x: stats.iqr(x))
        desc['Skewness'] = filtered_df[num_cols].apply(lambda x: stats.skew(x))
        st.dataframe(desc.round(2), use_container_width=True)
        
    with col2:
        st.subheader("Distribution Plot")
        plot_metric = st.selectbox("Feature to inspect:", num_cols, index=3)
        fig_dist = px.histogram(filtered_df, x=plot_metric, color='smoker', marginal="box", nbins=30, barmode="overlay")
        st.plotly_chart(fig_dist, use_container_width=True)
        
    st.subheader("Bivariate Correlation Matrix")
    corr = filtered_df[num_cols].corr()
    fig_corr = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu_r")
    st.plotly_chart(fig_corr, use_container_width=True)

with tab2:
    st.header("Interactive Hypothesis Testing Lab")
    
    test_type = st.radio("Select Hypothesis Test Protocol:", ["Two-Group Comparison", "One-Way ANOVA (Multi-Group)"])
    alpha = 0.05
    
    if test_type == "Two-Group Comparison":
        col1, col2 = st.columns(2)
        with col1:
            cat_var = st.selectbox("Grouping Variable (2-level):", ["smoker", "sex"])
            val1 = df[cat_var].unique()[0]
            val2 = df[cat_var].unique()[1]
        with col2:
            num_var = st.selectbox("Numerical Metric:", ["charges", "bmi", "age"])
            
        grp1 = df[df[cat_var] == val1][num_var]
        grp2 = df[df[cat_var] == val2][num_var]
        
        # Assumption checks
        s1 = stats.shapiro(grp1).pvalue
        s2 = stats.shapiro(grp2).pvalue
        lev = stats.levene(grp1, grp2).pvalue
        
        st.markdown(f"**Normality p-values:** `{val1}` = {s1:.3e}, `{val2}` = {s2:.3e}")
        st.markdown(f"**Levene's Equal Variance p-value:** {lev:.3e}")
        
        if s1 > alpha and s2 > alpha and lev > alpha:
            stat, pval = stats.ttest_ind(grp1, grp2)
            applied = "Two-Sample t-test"
        else:
            stat, pval = stats.mannwhitneyu(grp1, grp2)
            applied = "Mann-Whitney U Test (Non-parametric)"
            
        st.info(f"**Applied Protocol:** {applied}")
        st.write(f"**Test Statistic:** {stat:.4f} | **p-value:** {pval:.4e}")
        
        if pval < alpha:
            st.error(f"**Conclusion:** Reject H0 at alpha = 0.05. Significant difference found between {val1} and {val2}.")
        else:
            st.success(f"**Conclusion:** Fail to Reject H0 at alpha = 0.05. No significant difference detected.")
            
    else:
        st.subheader("One-Way ANOVA Test")
        group_var = st.selectbox("Categorical Factor (>=3 groups):", ["region", "children"])
        target_num = st.selectbox("Continuous Target:", ["charges", "bmi"])
        
        group_names = df[group_var].unique()
        arrays = [df[df[group_var] == g][target_num] for g in group_names]
        
        f_val, p_val = stats.f_oneway(*arrays)
        st.write(f"**F-Statistic:** {f_val:.4f} | **p-value:** {p_val:.4e}")
        
        if p_val < alpha:
            st.error("Reject H0: Mean differences across groups are statistically significant.")
        else:
            st.success("Fail to Reject H0: No statistically significant difference across groups.")

with tab3:
    st.header("Live Prediction & Gauss-Markov Residual Diagnostics")
    
    # Fit Model
    formula = "charges ~ age + bmi + children + C(sex) + C(smoker) + C(region)"
    model = smf.ols(formula=formula, data=df).fit()
    
    col1, col2 = st.columns([1, 1])
    
    with col1:
        st.subheader("Predict Medical Charges")
        in_age = st.slider("Age", 18, 65, 30)
        in_bmi = st.slider("BMI", 15.0, 50.0, 25.0)
        in_children = st.number_input("Children", min_value=0, max_value=5, value=0)
        in_sex = st.selectbox("Sex", df['sex'].unique())
        in_smoker = st.selectbox("Smoker", df['smoker'].unique())
        in_region = st.selectbox("Region", df['region'].unique())
        
        input_data = pd.DataFrame({
            'age': [in_age],
            'bmi': [in_bmi],
            'children': [in_children],
            'sex': [in_sex],
            'smoker': [in_smoker],
            'region': [in_region]
        })
        
        pred_res = model.get_prediction(input_data)
        pred_df = pred_res.summary_frame(alpha=0.05)
        
        st.markdown(f"### Predicted Charges: **${pred_df['mean'].iloc[0]:,.2f}**")
        st.markdown(f"**95% Confidence Interval:** [${pred_df['mean_ci_lower'].iloc[0]:,.2f}, ${pred_df['mean_ci_upper'].iloc[0]:,.2f}]")
        st.markdown(f"**95% Prediction Interval:** [${pred_df['obs_ci_lower'].iloc[0]:,.2f}, ${pred_df['obs_ci_upper'].iloc[0]:,.2f}]")
        
    with col2:
        st.subheader("Model Diagnostic Plots")
        fig_diag, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        
        ax1.scatter(model.fittedvalues, model.resid, alpha=0.3, edgecolors='k')
        ax1.axhline(0, color='red', linestyle='--')
        ax1.set_title("Residuals vs. Fitted")
        ax1.set_xlabel("Fitted Values")
        ax1.set_ylabel("Residuals")
        
        sm.qqplot(model.resid, line='45', fit=True, ax=ax2)
        ax2.set_title("Normal Q-Q")
        
        plt.tight_layout()
        st.pyplot(fig_diag)
        
        jb_stat, jb_pval, _, _ = sm.stats.stattools.jarque_bera(model.resid)
        st.caption(f"Jarque-Bera p-value: {jb_pval:.4e} (Non-normal residuals indicate potential non-linearities or interaction effects).")

Writing app.py
